# 02 超松弛迭代法与块迭代法

Jacobi 和 Gauss-Seidel 的收敛速度完全由原始分裂决定。超松弛迭代法（successive over-relaxation, SOR）在 Gauss-Seidel 更新上加入松弛因子，尝试改变误差衰减速度。块迭代法则把若干分量合并为一个小线性系统更新，适合矩阵具有自然分组结构的情形。


In [ ]:
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np

plt.rcParams["font.sans-serif"] = [
    "Arial Unicode MS",
    "PingFang SC",
    "Heiti SC",
    "SimHei",
    "Noto Sans CJK SC",
]
plt.rcParams["axes.unicode_minus"] = False

ROOT = pathlib.Path.cwd().resolve()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "pyproject.toml").exists() and (candidate / "src" / "py_sc").exists():
        ROOT = candidate
        break
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from py_sc import (
    block_gauss_seidel_iteration,
    block_jacobi_iteration,
    gauss_seidel_iteration,
    scan_sor_omega,
    sor_iteration,
    sor_iteration_matrix,
    spectral_radius,
)


## SOR 公式

Gauss-Seidel 的分量更新可记为

$$
\hat x_i^{(k+1)}=\frac{1}{a_{ii}}\left(b_i-\sum_{j<i}a_{ij}x_j^{(k+1)}-\sum_{j>i}a_{ij}x_j^{(k)}\right).
$$

SOR 不直接接受这个新值，而是令

$$
x_i^{(k+1)}=(1-\omega)x_i^{(k)}+\omega \hat x_i^{(k+1)}.
$$

当 $0<\omega<1$ 时称为欠松弛；当 $\omega=1$ 时退化为 Gauss-Seidel；当 $1<\omega<2$ 时称为超松弛。对 SPD 矩阵，$0<\omega<2$ 是常见的收敛范围，但最佳 $\omega$ 依赖矩阵结构。


In [ ]:
def teaching_sor(A, b, omega, x0=None, tolerance=1e-8, max_iterations=200):
    A = np.asarray(A, dtype=float)
    b = np.asarray(b, dtype=float)
    x = np.zeros_like(b) if x0 is None else np.asarray(x0, dtype=float).copy()
    residuals = [np.linalg.norm(b - A @ x) / np.linalg.norm(b)]
    for k in range(1, max_iterations + 1):
        old = x.copy()
        for i in range(len(b)):
            left = A[i, :i] @ x[:i]
            right = A[i, i + 1:] @ old[i + 1:]
            gs_value = (b[i] - left - right) / A[i, i]
            x[i] = (1 - omega) * old[i] + omega * gs_value
        residuals.append(np.linalg.norm(b - A @ x) / np.linalg.norm(b))
        if residuals[-1] <= tolerance:
            break
    return x, np.array(residuals)


## 松弛因子扫描

松弛因子太小会保守，太大可能震荡甚至发散。下面对一个 SPD 三对角系统扫描 $\omega$，观察迭代次数随参数变化的规律。


In [ ]:
A = np.array([
    [4.0, -1.0, 0.0, 0.0],
    [-1.0, 4.0, -1.0, 0.0],
    [0.0, -1.0, 4.0, -1.0],
    [0.0, 0.0, -1.0, 4.0],
])
b = np.array([1.0, 2.0, 3.0, 4.0])
omegas = np.linspace(0.5, 1.8, 27)
rows = scan_sor_omega(A, b, omegas, tolerance=1e-10, max_iterations=300)
iterations = np.array([row[1] if row[2] else np.nan for row in rows])

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(omegas, iterations, "o-")
ax.axvline(1.0, color="black", linestyle="--", linewidth=1, label="Gauss-Seidel")
ax.set_xlabel("松弛因子 omega")
ax.set_ylabel("达到容差的迭代次数")
ax.set_title("SOR 松弛因子扫描")
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()
print(rows[:5])


## SOR 迭代矩阵

SOR 的矩阵形式为

$$
(D+\omega L)x^{(k+1)}=\left[(1-\omega)D-\omega U\right]x^{(k)}+\omega b.
$$

因此迭代矩阵为

$$
B_{SOR}(\omega)=(D+\omega L)^{-1}\left[(1-\omega)D-\omega U\right].
$$

谱半径仍然给出定常迭代收敛速度的核心信息。


In [ ]:
radii = np.array([spectral_radius(sor_iteration_matrix(A, omega)) for omega in omegas])
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(omegas, radii, "s-", color="tab:red")
ax.axhline(1.0, color="black", linewidth=1)
ax.set_xlabel("松弛因子 omega")
ax.set_ylabel("谱半径")
ax.set_title("SOR 迭代矩阵谱半径")
ax.grid(True, alpha=0.3)
plt.show()


## 教学实现与正式实现对照

正式实现会检查 $0<\omega<2$、记录残差历史，并返回统一的结果对象。下面验证教学实现和正式实现的一致性。


In [ ]:
omega = 1.15
x_t, res_t = teaching_sor(A, b, omega, tolerance=1e-10)
formal = sor_iteration(A, b, omega=omega, tolerance=1e-10, max_iterations=300)
print("教学/正式解差异：", np.linalg.norm(x_t - formal.value))
print("最终残差：", formal.residual_norms[-1])
print("omega=1 是否接近 GS：", np.linalg.norm(sor_iteration(A, b, 1.0).value - gauss_seidel_iteration(A, b).value))


## 块 Jacobi 与块 Gauss-Seidel

如果矩阵按变量组自然分块，可以把标量更新换成小线性系统。设未知量分成若干块，块 Jacobi 在每一轮中使用上一轮的其它块：

$$
A_{ii}x_i^{(k+1)}=b_i-\sum_{j\ne i}A_{ij}x_j^{(k)}.
$$

块 Gauss-Seidel 对已经更新的前序块使用新值：

$$
A_{ii}x_i^{(k+1)}=b_i-\sum_{j<i}A_{ij}x_j^{(k+1)}-\sum_{j>i}A_{ij}x_j^{(k)}.
$$

块方法每一步成本更高，因为需要求解小块系统；但当块内耦合强、块间耦合弱时，常能减少迭代次数。


In [ ]:
def teaching_block_jacobi(A, b, block_sizes, tolerance=1e-8, max_iterations=200):
    A = np.asarray(A, dtype=float)
    b = np.asarray(b, dtype=float)
    x = np.zeros_like(b)
    starts = np.cumsum([0, *block_sizes[:-1]])
    blocks = [slice(start, start + size) for start, size in zip(starts, block_sizes)]
    residuals = [np.linalg.norm(b - A @ x) / np.linalg.norm(b)]
    for k in range(max_iterations):
        old = x.copy()
        new = old.copy()
        for block in blocks:
            rhs = b[block] - A[block, :] @ old + A[block, block] @ old[block]
            new[block] = np.linalg.solve(A[block, block], rhs)
        x = new
        residuals.append(np.linalg.norm(b - A @ x) / np.linalg.norm(b))
        if residuals[-1] <= tolerance:
            break
    return x, np.array(residuals)


In [ ]:
block_sizes = [2, 2]
x_block, res_block = teaching_block_jacobi(A, b, block_sizes, tolerance=1e-10)
formal_block_j = block_jacobi_iteration(A, b, block_sizes, tolerance=1e-10)
formal_block_gs = block_gauss_seidel_iteration(A, b, block_sizes, tolerance=1e-10)
print("块 Jacobi 教学/正式差异：", np.linalg.norm(x_block - formal_block_j.value))
print("块 Jacobi 迭代次数：", formal_block_j.iterations)
print("块 GS 迭代次数：", formal_block_gs.iterations)


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.semilogy(formal_block_j.residual_norms, "o-", label="块 Jacobi")
ax.semilogy(formal_block_gs.residual_norms, "s-", label="块 Gauss-Seidel")
ax.semilogy(sor_iteration(A, b, 1.15, tolerance=1e-10).residual_norms, "^-", label="SOR omega=1.15")
ax.set_xlabel("迭代次数")
ax.set_ylabel("相对残差")
ax.set_title("块迭代与 SOR 残差比较")
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()


## 适用条件和失败情形

SOR 的效率高度依赖 $\omega$。过小的 $\omega$ 可能比 Gauss-Seidel 更慢，过大的 $\omega$ 可能让残差震荡。块方法适合变量按物理量、网格线或子区域自然分组的系统；若块太大，每轮成本会变高，若块划分不符合耦合结构，则不一定带来收益。


## 本节小结

SOR 在 Gauss-Seidel 基础上引入松弛因子，$\omega=1$ 时退化为 Gauss-Seidel。松弛因子的选择能显著影响收敛速度，因此参数扫描和谱半径分析都很有用。块 Jacobi 和块 Gauss-Seidel 用小线性系统替代标量更新，适合有自然分块结构的问题。实际使用时应同时考虑迭代次数和每轮计算成本。
